# Proteus on Kaggle

Stabilizing designed proteins with mechanism-level strategies, then adjudicating
the result with Rosetta and a refold check.

**Why this runs here and not on a laptop.** PyRosetta ships no Windows wheel, and
ESMFold needs roughly 16 GB of RAM on CPU because `esmfold_v1` bundles ESM-2 3B.
Kaggle gives you Linux, enough memory, and a GPU that turns ESMFold from minutes
into seconds.

**Runtime:** set Accelerator to GPU if you want step 6. Steps 1-5 are fine on CPU.


## 1. Install


In [ ]:
# PyRosetta: free for academic use, licence required for commercial use.
!pip install -q pyrosetta-installer
import pyrosetta_installer
pyrosetta_installer.install_pyrosetta()


In [ ]:
# Proteus itself. Either point at your repo:
#   !pip install -q git+https://github.com/<you>/proteus
# or upload the package as a Kaggle Dataset and install from disk:
!pip install -q /kaggle/input/proteus-src

# Optional, for the refold gate in step 6 (needs a GPU runtime):
# !pip install -q 'transformers>=4.35' accelerate


## 2. Load and diagnose

Nothing is modified here. This reports which stabilization mechanisms the fold
actually admits, and how many sites each one finds.


In [ ]:
import glob
from proteus import DesignContext, REGISTRY, from_pdb
from proteus import membrane as membrane_mod
from proteus.scoring import HeuristicScorer

PDB = next(iter(glob.glob('/kaggle/input/**/*.pdb', recursive=True)))
IS_MEMBRANE = False          # True inverts the burial rules inside the bilayer
FROZEN = frozenset()         # e.g. frozenset(range(1, 11)) to hold a binding face

structure = from_pdb(PDB)
membrane = membrane_mod.estimate(structure) if IS_MEMBRANE else None
ctx = DesignContext(structure=structure, frozen=FROZEN, membrane=membrane)

print(ctx.summary())
print()
print(HeuristicScorer().score(ctx, structure.sequence).table())
print()
for s in REGISTRY.applicable(ctx):
    print(f'{s.name:<26} {len(s.diagnose(ctx)):3d} sites')


## 3. Run the loop

Cheap exploration. A bandit picks which mechanisms to try, Metropolis decides what
to keep, and the temperature is calibrated from the observed score scale.


In [ ]:
from proteus.engine import Engine

engine = Engine(ctx, seed=0, protein=PDB.split('/')[-1])
result = engine.run(generations=60, verbose=True)

print()
print(result.summary())


## 4. Why each change was made

Every mutation traces back to the mechanism that proposed it. This is the point of
working at mechanism level rather than per-mutation.


In [ ]:
print(result.explain(limit=30))
print()
for name, n in sorted(result.credit().items(), key=lambda kv: -kv[1]):
    print(f'{name:<26} {n} surviving mutations')


## 5. Rosetta adjudication

The heuristic objective explores; a real energy function decides. Rosetta runs on
the finalist rather than inside the loop because a FastRelax costs seconds to
minutes per call.

A negative delta means Rosetta agrees. A positive one means the two objectives
disagree, which is information worth having rather than a failure to hide.


In [ ]:
from proteus.backends import rosetta

resolution = rosetta.build_resolution_from_sequence(ctx, result.best_sequence)
print(f'{len(resolution.allowed)} positions to apply')

backend = rosetta.RosettaBackend(membrane=IS_MEMBRANE)
print(f'energy function: {backend.weights}')

refined = backend.refine(PDB, resolution, structure, frozen=FROZEN,
                         out_pdb='/kaggle/working/proteus_refined.pdb')
print(refined.describe())
print()
print('Rosetta', 'AGREES with' if refined.delta < 0 else 'DISAGREES with',
      'the heuristic objective.')


## 6. Refold self-consistency (GPU runtime)

The check everything else is conditional on: does the designed sequence still fold
to the backbone the strategies were reasoning about? RMSD is superposed with Kabsch
first, including a determinant correction so a mirror image cannot pass.


In [ ]:
from proteus.validate import ESMFoldGate

check = ESMFoldGate(rmsd_cutoff=2.0).check(result.best_sequence, structure)
print(check.describe())
if not check.passed:
    print('The sequence does not refold to the intended backbone.')
    print('Treat the score improvement as unverified.')


## 7. Accumulate knowledge across runs

Tag each run with a structural fingerprint and keep it, so the next protein starts
with a prior from structurally similar folds instead of from scratch. Save the
knowledge base to `/kaggle/working` and re-upload it as a dataset input next time.


In [ ]:
from proteus import KnowledgeBase
from pathlib import Path

KB = Path('/kaggle/working/proteus_kb.json')
prior = Path('/kaggle/input/proteus-kb/proteus_kb.json')
kb = KnowledgeBase.load(prior if prior.exists() else KB)
print(f'knowledge base: {len(kb)} observations')

engine2 = Engine(ctx, seed=1, knowledge=kb, protein=PDB.split('/')[-1])
if engine2.fingerprint is not None:
    print(engine2.fingerprint.describe())
    for sim, name in kb.neighbours(engine2.fingerprint, k=3):
        print(f'  similar: {name} ({sim:.2f})')

result2 = engine2.run(generations=60)
kb.save(KB)
print()
print(kb.report(engine2.fingerprint, top=8))


---

### Notes

- `HeuristicScorer` is a screening function, not a force field. Improvements against
  it are not stability predictions; that is what step 5 is for.
- Credit is joint: when two mechanisms are applied together and the result improves,
  both are rewarded. Raw per-arm history is retained so the ambiguity is analysable.
- For a membrane protein, prefer an OPM-oriented structure. The bilayer estimator
  locates the membrane from exposed hydrophobicity, so a design whose surface is
  already wrong will mislocate it.
